# 15주 · 종합 검토·검산 가능한 AI 요청

수치해석및실습 CE273 · 2027학년도 1학기

원본 슬라이드의 개념·예제·차시 순서에 연결한 **신규 병행 실습**. 원본 코드의 단순 복사본과 구분

**목표**: 수치 원리·도구 기능 이해 → 좋은 AI 질문·지시 → 오차·수렴·결과의 검토

**수업**: 2027-06-08 종합 복습 · 대면 / 2027-06-10 기말평가 예정 · 평가 예정

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE273-numerical-analysis/week15/CE273_W15.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE273-numerical-analysis/week15)

슬라이드 예측 → 작은 손계산 → 실행 → 한 조건 변경 → AI 요청 수정 → 단위·오차·수렴·실패 상태 검토

셀 실행: Shift+Enter. 처음에는 위에서 아래로 실행. 공통 준비 셀 실행 후 각 절의 개별 실행도 가능. None 항목과 변경 셀은 직접 작성

## 0. 실행 환경·보조 함수

필요한 버전만 설치. 업데이트한 패키지를 이미 불러온 경우 런타임 재시작 후 처음부터 실행

보조 함수의 세부 문법 암기 불필요. 입력·처리·반환값·오차 기준·실패 조건부터 확인. 아래 실습의 호출 코드와 슬라이드 식 대조

In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.11", "matplotlib": "3.8", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, linalg, interpolate, integrate

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 자료·최소 원소 수 확인 필요")
    return x

def relative_error(truth, estimate):
    # 기준값 0에서 상대오차를 정의하지 않음. 절대오차 별도 확인
    return np.nan if truth == 0 else abs((truth-estimate)/truth)
def root_result(x, history, converged, reason):
    return {"root":float(x),"history":pd.DataFrame(history),"iterations":len(history),"converged":converged,"reason":reason}

def root_options(xtol,ftol,max_iter):
    if not np.isfinite([xtol,ftol]).all() or xtol<=0 or ftol<=0 or not isinstance(max_iter,int) or max_iter<=0:
        raise ValueError("양의 위치·잔차 허용오차, 양의 정수 최대 반복 수 필요")

def bracket_root(f,a,b,method="bisection",xtol=1e-10,ftol=1e-10,max_iter=200):
    root_options(xtol,ftol,max_iter)
    if method not in ["bisection","false_position","illinois"] or not np.isfinite([a,b]).all() or a>=b:raise ValueError("방법·a<b 확인 필요")
    fa,fb=float(f(a)),float(f(b));history=[]
    if not np.isfinite([fa,fb]).all():raise ValueError("유한한 양 끝 함수값 필요")
    if fa==0:return root_result(a,history,True,"left endpoint")
    if fb==0:return root_result(b,history,True,"right endpoint")
    if np.signbit(fa)==np.signbit(fb):raise ValueError("연속성 확인 후 부호 변화 구간 필요")
    wa,wb=fa,fb;previous=None;retained=None;x=(a+b)/2
    for k in range(max_iter):
        x=(a+b)/2 if method=="bisection" else b-wb*(b-a)/(wb-wa)
        fx=float(f(x))
        if not np.isfinite(fx):raise ValueError("반복 중 비유한 함수값")
        step=np.inf if previous is None else abs(x-previous)
        history.append({"iteration":k+1,"a":a,"b":b,"x":x,"residual":abs(fx),"step":step})
        if fx==0 or (abs(fx)<=ftol and (b-a<=2*xtol if method=="bisection" else step<=xtol)):
            return root_result(x,history,True,"position and residual")
        if np.signbit(fa)!=np.signbit(fx):
            b,fb,wb=x,fx,fx
            wa=wa/2 if method=="illinois" and retained=="a" else fa
            retained="a"
        else:
            a,fa,wa=x,fx,fx
            wb=wb/2 if method=="illinois" and retained=="b" else fb
            retained="b"
        previous=x
    return root_result(x,history,False,"maximum iterations")

def open_root(f,x0,method="newton",df=None,g=None,x1=None,xtol=1e-10,ftol=1e-10,max_iter=100):
    root_options(xtol,ftol,max_iter)
    if method not in ["newton","fixed_point","secant"] or not np.isfinite(x0):raise ValueError("방법·초기값 확인 필요")
    if method=="newton" and not callable(df):raise ValueError("도함수 필요")
    if method=="fixed_point" and not callable(g):raise ValueError("고정점 함수 필요")
    if method=="secant" and (x1 is None or not np.isfinite(x1) or x1==x0):raise ValueError("서로 다른 두 유한 초기값 필요")
    previous=float(x0);current=float(x1 if method=="secant" else x0);history=[]
    if float(f(current))==0:return root_result(current,history,True,"initial root")
    for k in range(max_iter):
        fc=float(f(current))
        if method=="fixed_point":nxt=float(g(current))
        elif method=="newton":
            derivative=float(df(current))
            if not np.isfinite(derivative) or abs(derivative)<np.finfo(float).eps:raise ValueError("0 또는 작은 도함수")
            nxt=current-fc/derivative
        else:
            fp=float(f(previous));den=fc-fp
            if not np.isfinite(den) or abs(den)<np.finfo(float).eps:raise ValueError("0 또는 작은 함수값 차이")
            nxt=current-fc*(current-previous)/den
        if not np.isfinite(nxt):raise ValueError("비유한 반복값")
        residual=abs(float(f(nxt)));step=abs(nxt-current)
        if not np.isfinite(residual):raise ValueError("비유한 잔차")
        history.append({"iteration":k+1,"x":nxt,"residual":residual,"step":step})
        if residual==0 or (residual<=ftol and step<=xtol):return root_result(nxt,history,True,"position and residual")
        previous,current=current,nxt
    return root_result(current,history,False,"maximum iterations")
def ode_steps(f, start, y0, end, h, method="euler"):
    if not np.isfinite([start,y0,end,h]).all() or end<=start or h<=0:raise ValueError("유한한 초기조건·양의 구간·양의 h 필요")
    count=(end-start)/h;n=int(round(count))
    if not 1<=n<=100000 or not np.isclose(count,n,rtol=1e-12,atol=1e-12):raise ValueError("h로 구간을 정확히 분할할 양의 정수 스텝 수 필요")
    if method not in ["euler","heun","midpoint","ralston34","rk4"]:raise ValueError("방법 확인 필요")
    x=np.linspace(start,end,n+1);y=np.empty(n+1);y[0]=y0
    for i in range(n):
        k1=f(x[i],y[i])
        if method=="euler":slope=k1
        elif method=="heun":slope=(k1+f(x[i]+h,y[i]+h*k1))/2
        elif method=="midpoint":slope=f(x[i]+h/2,y[i]+h*k1/2)
        elif method=="ralston34":slope=k1/3+2*f(x[i]+3*h/4,y[i]+3*h*k1/4)/3
        else:
            k2=f(x[i]+h/2,y[i]+h*k1/2);k3=f(x[i]+h/2,y[i]+h*k2/2);k4=f(x[i]+h,y[i]+h*k3)
            slope=(k1+2*k2+2*k3+k4)/6
        y[i+1]=y[i]+h*slope
        if not np.isfinite(y[i+1]):raise ValueError("비유한 수치해: 스텝·모형·스케일 점검")
    return x,y
def linear_system(A, b):
    A=np.asarray(A,dtype=float);b=vector(b)
    if A.ndim!=2 or A.shape[0]!=A.shape[1] or A.shape[0]!=len(b) or not np.isfinite(A).all():
        raise ValueError("정방행렬·차원·유한한 값 확인 필요")
    x=np.linalg.solve(A,b)
    residual=A@x-b
    den=np.linalg.norm(A,np.inf)*np.linalg.norm(x,np.inf)+np.linalg.norm(b,np.inf)
    return {"x":x,"residual":residual,"backward_error":0. if den==0 else np.linalg.norm(residual,np.inf)/den,"condition_2":np.linalg.cond(A)}

def gaussian_pivot(A,b):
    A=np.asarray(A,dtype=float).copy();b=vector(b).copy()
    if A.ndim!=2 or A.shape!=(len(b),len(b)) or not np.isfinite(A).all():raise ValueError("정방행렬·우변 차원 확인 필요")
    n=len(b)
    for k in range(n):
        pivot=k+np.argmax(abs(A[k:,k]))
        if abs(A[pivot,k])<=np.finfo(float).eps*np.linalg.norm(A,np.inf):raise np.linalg.LinAlgError("0 또는 작은 피벗: 특이성·스케일 검토")
        A[[k,pivot]]=A[[pivot,k]];b[[k,pivot]]=b[[pivot,k]]
        for i in range(k+1,n):
            factor=A[i,k]/A[k,k];A[i,k:]-=factor*A[k,k:];b[i]-=factor*b[k]
    x=np.zeros(n)
    for i in range(n-1,-1,-1):x[i]=(b[i]-A[i,i+1:]@x[i+1:])/A[i,i]
    return x


## S01. 방법 선택과 실패 보고

**슬라이드 연결**: 2026_S1_NA_Wk15a_Review.pptx p4, 5, 6, 7, 8, 9, 10, 11, 12

**자료 구분**: 원본 예제·수치·원리에 기반. 코드에 표시한 추가 검토는 조건 변경 실험

**먼저 예측**: 같은 숫자가 나왔다는 이유만으로 모형과 알고리즘이 모두 옳은가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
A=np.array([[2.,-1],[-1,2]]);b=np.array([0.,3]);linear=linear_system(A,b)
f=lambda x:np.exp(-x)-x;root=open_root(f,0,"newton",df=lambda x:-np.exp(-x)-1)
print({"linear_backward_error":linear["backward_error"],"root":root["root"],"root_converged":root["converged"]})
assert linear["backward_error"]<1e-14 and root["converged"]
print("보고: 문제·자료·단위·가정·방법·허용오차·수렴/실패·독립 검산·한계")

**조건 변경**: 입력·초기값·스텝·허용오차 중 하나를 바꾼 사본과 결과 비교

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**AI 요청에 넣을 정보**: 목적·입력·단위·제약·알고리즘·출력·검증 기준·실패 보고를 포함한 요청 작성

**결과 검토**: 채택·수정·거절한 AI 결과와 그 근거를 함께 남김

예상·실행·독립 검산·수정 지시·채택 또는 거절 이유 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## 주차 확인

최초 AI 요청·수정 지시·독립 검산·최종 판단 제출

입력 항목:

- `실행_성공만으로_수치적_정확성_입증`: 판단 · Python `True`/`False`
- `실패_상태도_보고_대상`: 판단 · Python `True`/`False`
- `해석에는_모형의_가정과_한계가_필요`: 판단 · Python `True`/`False`

공개 계산·개념 확인용 항목. 값만 복사하면 학습 완료로 판단 불가. 가정·적용 범위·검산 근거는 아래 서술과 함께 확인

In [ ]:
answers = {'실행_성공만으로_수치적_정확성_입증': None, '실패_상태도_보고_대상': None, '해석에는_모형의_가정과_한계가_필요': None}


In [ ]:
def check_answers(values):
    expected = {'실행_성공만으로_수치적_정확성_입증': False, '실패_상태도_보고_대상': True, '해석에는_모형의_가정과_한계가_필요': True}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## AI 요청·검토 기록

1. 문제·모형·입력·단위·자료 출처
2. 최초 요청: 목적·조건·알고리즘·출력·검증 기준
3. 실행 전 예측·작은 손계산의 근거
4. 의심한 출력·오차·수렴·실패 조건과 수정 지시
5. 채택·수정·거절한 내용, 독립 검산·한계
6. AI 없이 설명 가능한 원리 한 가지

글자 수 자동 채점 없음. 실행 성공·작은 갱신량·작은 잔차와 문제 해결의 타당성 구분

In [ ]:
review_log = {
    "문제·모형·자료·단위": None,
    "최초 AI 요청": None,
    "예측·손계산": None,
    "의심한 부분·수정 지시": None,
    "검산·최종 판단·한계": None,
    "내 설명": None,
}
